# Integración del comité técnico (Task 2 y Task 3)

En este notebook se reúnen las siete secciones de los notebooks `01` a `07`, se concilian las cifras que no cuadran, se elige el umbral y la capacidad, y se redacta el memo final. Al final se indica cómo preparar la presentación y la defensa.

| Rol | Integrante | Notebook |
|---|---|---|
| 1. Modelado | Majo | `01_rol1_modelado.ipynb` |
| 2. Transferencia | Tunchez | `02_rol2_transferencia.ipynb` |
| 3. Series de tiempo | Dulce | `03_rol3_series_tiempo.ipynb` |
| 4. Modelos generativos | Nadissa | `04_rol4_generativos.ipynb` |
| 5. Lenguaje natural | Javier | `05_rol5_lenguaje.ipynb` |
| 6. Explicabilidad | Chet | `06_rol6_explicabilidad.ipynb` |
| 7. Operaciones y cumplimiento | Mejía | `07_rol7_operaciones.ipynb` |

Se agrega la raíz del repositorio al `sys.path` para importar los módulos de `src/`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import math
import pandas as pd
from src import case_data as cd
from src.metrics import expected_counts
from src.operations import team_capacity, threshold_table

## Task 2.1. Revisión conjunta de las secciones

### a. Cifra clave y dependencias de cada rol

| Rol | Cifra clave | Recomendación | Depende de |
|---|---|---|---|
| 1 | Precisión de U2 de 3.85% a prevalencia real (0.95 en el conjunto balanceado) | Reentrenar con detención temprana, regularización y BCE ponderada | Rol 3 (tipo de partición) |
| 2 | AUC de 0.87 con ajuste fino de dos bloques | Ajuste fino de los dos últimos bloques | Rol 3 (tipo de partición) |
| 3 | AUC de 0.81 con partición temporal | Usar 0.81 y ventanas de 30 días | Roles 1 y 2 (cifras que corrige) |
| 4 | AUC temporal de 0.81 a 0.83 con difusión | Difusión con filtro de privacidad | Rol 3 (AUC temporal base) |
| 5 | Alucinación de 5% y 4.5 minutos ahorrados por narrativa | Modelo de lenguaje solo como borrador | Rol 7 (volumen de alertas a narrar) |
| 6 | Error de eficiencia de SHAP de 3e-07 | Entregar SHAP, no LIME de 500 muestras ni atención | Ninguna |
| 7 | 250 alertas diarias frente a una capacidad de 40 | No operar con U2 y el equipo actual | Rol 1 (umbral U2) |

### b. Cifras que no cuadran

**Precisión de 0.95 (Rol 1) frente a 4% de casos confirmados en el piloto (Rol 7).** Se proyecta U2 a los 6,000 remitentes diarios con prevalencia de 0.2% y se compara con el piloto.

In [2]:
u2 = cd.thresholds().set_index("umbral").loc["U2"]
counts = expected_counts(u2["recall"], u2["fpr"], cd.DAILY_SENDERS, cd.PREVALENCE)
comparison = pd.DataFrame({
    "U2 proyectado a prevalencia real": [counts["alertas"], counts["tp"], counts["tp"] / counts["alertas"]],
    "Piloto (Rol 7)": [cd.PILOT_DAILY_ALERTS, cd.PILOT_DAILY_ALERTS * cd.PILOT_CONFIRMED_RATE, cd.PILOT_CONFIRMED_RATE],
    "Tabla balanceada (Rol 1)": [None, None, u2["precision_balanceada"]],
}, index=["alertas_diarias", "casos_detectados", "precision"])
comparison.round(4)

,U2 proyectado a prevalencia real,Piloto (Rol 7),Tabla balanceada (Rol 1)
alertas_diarias,249.1200,250.00,NaN
casos_detectados,9.6000,10.00,NaN
precision,0.0385,0.04,0.95


La proyección de U2 da 249.12 alertas, 9.6 casos detectados y una precisión de 0.0385. El piloto reporta 250 alertas, 10 casos y 0.04. Las dos cifras son ciertas a la vez. La precisión de 0.95 se midió en un conjunto con 50% de positivos, mientras que en la operación solo el 0.2% lo es. Con el mismo recall y la misma tasa de falsos positivos, la precisión cae porque los 5,988 remitentes normales generan muchos más falsos positivos que los 12 casos reales. **La cifra correcta para decidir es la del piloto (cerca de 4%)**, y la proyección la confirma. Esto obliga a elegir el umbral con la tabla proyectada de la Task 2.2 y no con la precisión balanceada.

In [3]:
print(f"Recall de U2 en la tabla del Rol 1: {u2['recall']:.2f}")
print(f"Recall observado en el piloto: {cd.PILOT_DAILY_ALERTS * cd.PILOT_CONFIRMED_RATE / cd.EXPECTED_DAILY_CASES:.2f}")
print(f"AUC partición aleatoria (Roles 1 y 2): {cd.AUC_RANDOM_SPLIT} | partición temporal (Rol 3): {cd.AUC_TEMPORAL_SPLIT}")

Recall de U2 en la tabla del Rol 1: 0.80
Recall observado en el piloto: 0.83
AUC partición aleatoria (Roles 1 y 2): 0.87 | partición temporal (Rol 3): 0.81


**AUC de 0.87 (Roles 1 y 2) frente a 0.81 (Rol 3).** Ambas cifras son ciertas para su forma de validar. El 0.87 se obtuvo con partición aleatoria de ventanas, que filtra información del mismo remitente y de meses futuros. El 0.81 se obtuvo entrenando con el pasado y validando con el futuro, como ocurrirá en producción. **La cifra correcta es 0.81.** Esto afecta la decisión porque el recall y la tasa de falsos positivos de la tabla del Rol 1 también provienen de partición aleatoria, por lo que los casos detectados de la Task 2.2 son una cota optimista.

El recall de U2 en la tabla (0.80) y el observado en el piloto (0.83) sí son consistentes, dado que el piloto reporta cifras aproximadas.

### c. Cifras optimistas

- **Precisiones de 0.90 a 0.99 (Rol 1).** Se midieron con validación balanceada.
- **AUC de 0.87 y tabla de estrategias (Rol 2).** Se midieron con partición aleatoria.
- **Recall y tasa de falsos positivos de la tabla de umbrales (Rol 1).** Comparten la partición aleatoria, por lo que el recall real probablemente es menor.
- **AUC de 0.83 con sintéticos (Rol 4).** Es una sola comparación con una ganancia de 0.02, y el 3% de sintéticos casi idénticos a clientes reales puede filtrar información de esos clientes.
- **83.3% de casos detectados en el piloto (Rol 7).** Supone revisar las 250 alertas diarias, algo que el equipo no puede hacer.
- **4.5 minutos ahorrados (Rol 5).** Provienen de una auditoría de 40 narrativas, y el 5% con hechos inventados exige correcciones adicionales.

## Task 2.2. Decisión con los datos reconciliados

### a. Tabla de umbrales a prevalencia real

Se proyecta cada umbral a 6,000 remitentes diarios con prevalencia de 0.2%. El costo de revisión usa USD 12 por alerta y el costo de no detección usa USD 50,000 por caso. La columna `analistas_necesarios` divide las alertas entre 5 revisiones por analista y redondea hacia arriba.

In [4]:
table = threshold_table(
    cd.thresholds(), cd.DAILY_SENDERS, cd.PREVALENCE, cd.REVIEW_COST_USD, cd.MISSED_CASE_COST_USD
)
table["analistas_necesarios"] = [math.ceil(a / cd.ALERTS_PER_ANALYST) for a in table["alertas_diarias"]]
table.round(3)

,umbral,alertas_diarias,detectados,no_detectados,precision_esperada,costo_revision_usd,costo_no_deteccion_usd,costo_total_usd,analistas_necesarios
0,U1,609.60,10.8,1.2,0.018,7315.20,60000.0,67315.20,122
1,U2,249.12,9.6,2.4,0.039,2989.44,120000.0,122989.44,50
2,U3,67.68,7.8,4.2,0.115,812.16,210000.0,210812.16,14
3,U4,35.94,6.0,6.0,0.167,431.28,300000.0,300431.28,8


Para comparar umbrales vecinos se calcula cuánto cuesta en revisiones cada caso adicional detectado al bajar el umbral.

In [5]:
ordered = table.iloc[::-1].reset_index(drop=True)
marginal = pd.DataFrame({
    "cambio": [f"{a} -> {b}" for a, b in zip(ordered["umbral"], ordered["umbral"].iloc[1:])],
    "revisiones_extra": ordered["alertas_diarias"].diff().iloc[1:].to_numpy(),
    "casos_extra": ordered["detectados"].diff().iloc[1:].to_numpy(),
})
marginal["costo_por_caso_extra_usd"] = marginal["revisiones_extra"] * cd.REVIEW_COST_USD / marginal["casos_extra"]
marginal.round(1)

,cambio,revisiones_extra,casos_extra,costo_por_caso_extra_usd
0,U4 -> U3,31.7,1.8,211.6
1,U3 -> U2,181.4,1.8,1209.6
2,U2 -> U1,360.5,1.2,3604.8


In [6]:
capacity = team_capacity(cd.ANALYSTS, cd.ALERTS_PER_ANALYST)
chosen = table.set_index("umbral").loc["U3"]
analysts = int(chosen["analistas_necesarios"])
print(f"Capacidad actual: {capacity} alertas/día")
print(f"U3: {chosen['alertas_diarias']:.1f} alertas/día, {chosen['detectados']:.1f} casos detectados "
      f"({chosen['detectados'] / cd.EXPECTED_DAILY_CASES:.0%} de {cd.EXPECTED_DAILY_CASES})")
print(f"Analistas necesarios para U3: {analysts} ({analysts - cd.ANALYSTS} adicionales), "
      f"capacidad {team_capacity(analysts, cd.ALERTS_PER_ANALYST)} alertas/día")

Capacidad actual: 40 alertas/día
U3: 67.7 alertas/día, 7.8 casos detectados (65% de 12)
Analistas necesarios para U3: 14 (6 adicionales), capacidad 70 alertas/día


### b. Umbral y capacidad elegidos

Se elige **U3 con 14 analistas, es decir, una capacidad de 70 alertas diarias** (6 analistas adicionales).

- **Costo de no detectar.** Bajar el umbral siempre conviene en dinero. Cada caso adicional cuesta USD 212 en revisiones al pasar de U4 a U3, USD 1,210 de U3 a U2 y USD 3,605 de U2 a U1, todos muy por debajo de los USD 50,000 de un caso no detectado. Por eso U1 tiene el menor costo total (USD 67,315 diarios).
- **Capacidad del equipo.** El límite real es la capacidad. U1 necesita 122 analistas y U2 necesita 50, frente a los 8 actuales. No es posible contratarlos para el lunes.
- **Costo de revisar.** U4 cabe en el equipo actual (36 alertas, USD 431 diarios), pero deja sin detectar 6 de los 12 casos diarios. U3 genera 67.7 alertas con un costo de USD 812 diarios, detecta 7.8 casos (65%) y solo requiere 6 analistas más.

U3 es el umbral más bajo que se puede operar con un aumento de equipo alcanzable. El paso siguiente sería U2, cuando la capacidad crezca y la validación temporal confirme el recall.

### c. Decisión

**Lanzar con condiciones.** Las condiciones medibles son las siguientes.

1. **Modelo validado.** El modelo se reentrena con detención temprana y BCE ponderada, y antes de activar alertas obtiene un AUC con partición temporal de al menos 0.81. La tabla de umbrales se recalcula con esa partición y con prevalencia real.
2. **Capacidad suficiente.** Umbral U3 con 14 analistas. Las alertas diarias no superan 70 y no quedan alertas pendientes al cierre de cada día. La cola se ordena por puntaje.
3. **Desempeño mensual.** El recall sobre casos confirmados es de al menos 60% y la precisión de al menos 10%. Si alguna baja de ese valor, se recalibra el umbral.
4. **Explicación y narrativa.** El 100% de las alertas lleva explicación SHAP y una narrativa revisada y firmada por un oficial. La auditoría mensual de 40 narrativas muestra una alucinación de 5% o menos.
5. **Privacidad.** Los sintéticos de difusión solo se usan si, tras el filtro de distancia, no queda ninguna secuencia casi idéntica a un cliente real.

## Task 2.3. Dónde falló la IA

> **Borrador.** El caso 1 ocurrió al preparar este notebook. Los casos 2 y 3 son errores que la IA comete con facilidad en este examen. Cada integrante debe confirmarlos o sustituirlos por los que el grupo observó.

| Caso | Lo que dijo la IA | Supuesto que violaba | Cómo se verificó |
|---|---|---|---|
| 1 | Al extraer el texto del enunciado en PDF, la fila de SHAP salió como códigos de glifos ilegibles (`<0375><0D48>...`) y varias palabras salieron cortadas. | Que el texto extraído del PDF es igual al documento. | Se decodificaron los glifos con la tabla de caracteres de la fuente y con la ecuación incrustada en el PDF. El valor es 3 × 10⁻⁷. |
| 2 | Con U2 habría unas 10 alertas diarias, porque la precisión es 0.95. | Que la prevalencia del conjunto de validación (50%) es la misma de la operación (0.2%). | La proyección de la Task 2.1 da 249.12 alertas y una precisión de 0.0385, igual que el piloto. |
| 3 | Se debe elegir U1 porque tiene el menor costo total (USD 67,315). | Que la capacidad de revisión es ilimitada. | La columna `analistas_necesarios` de la Task 2.2 muestra que U1 requiere 122 analistas, frente a 8. |

Como la IA sí se equivocó, el inciso b no aplica.

## Task 2.4. Memo final

### a. Defensa ante la autoridad reguladora

Ante la autoridad reguladora se presentaría la validación con partición temporal (AUC de 0.81 en abril y mayo), la tabla de umbrales proyectada a la prevalencia real y la explicación SHAP de cada alerta, con un error de eficiencia de 3e-07. Se reconoce como riesgo residual que con U3 se esperan 4.2 casos no detectados por día, cifra que puede ser mayor porque el recall proviene de una partición aleatoria. La empresa se compromete a medir cada mes el recall, la precisión y las alertas frente a la capacidad, y a recalibrar el umbral si se incumple alguna condición. Responde por la decisión el oficial de cumplimiento, con la aprobación de la gerencia general.

### b. Memo

---

**Memo del comité técnico. RemesaSegura S.A. Sistema de detección de lavado de dinero.**

**Decisión.** Lanzar con condiciones el lunes. Se usa el umbral U3 con 14 analistas (capacidad de 70 alertas diarias) para detectar 7.8 de los 12 casos diarios esperados, con un costo de revisión de USD 812 diarios. El modelo se reentrena y se valida con partición temporal antes de activar las alertas.

**1. Modelado (Majo).** Se recomienda no usar el modelo de la época 20. Su pérdida de validación subió de 0.21 (época 6) a 0.34, lo que indica sobreajuste. El modelo debe reentrenarse con detención temprana, regularización y entropía cruzada ponderada (peso positivo de 499). Además, la precisión de la tabla proviene de un conjunto balanceado. A la prevalencia real, U2 tiene una precisión de 3.85% y no de 95%, por lo que todas las métricas deben recalcularse con la proporción real.

**2. Transferencia (Tunchez).** Se recomienda el ajuste fino de los dos últimos bloques con tasa de aprendizaje 10 veces menor. Obtiene el mayor AUC (0.87) con 18% de parámetros entrenables y 1.5 GPU-horas, frente a 6 GPU-horas del ajuste completo, que sobreajusta los 1,200 positivos y baja a 0.84. Antes de usarlo debe homologarse la normalización de montos con la del socio. El AUC de 0.87 se obtuvo con partición aleatoria y debe confirmarse con partición temporal (Rol 3).

**3. Series de tiempo (Dulce).** Las cifras de AUC de los Roles 1 y 2 están infladas por la partición aleatoria de ventanas, que mezcla ventanas del mismo remitente y de meses futuros entre entrenamiento y validación. Con partición temporal el AUC baja de 0.87 a 0.81, y esta es la cifra que debe usar la gerencia. Se recomienda una ventana de 30 días, que contiene los ciclos de 7, 15 y 30 días. La exactitud no debe reportarse, porque un sistema que nunca alerta logra 99.8%.

**4. Modelos generativos (Nadissa).** Se recomienda usar sintéticos solo del modelo de difusión, que cubre 6 de 7 tipologías y sube el AUC temporal de 0.81 a 0.83. La GAN se descarta por colapso de modos (2 de 7) y el autoencoder variacional por generar secuencias borrosas. Antes de usarlos se deben eliminar los casi duplicados de clientes reales (3% de los sintéticos) con un filtro de distancia. Los sintéticos se usan solo para entrenar y el desempeño se mide únicamente con datos reales de abril y mayo.

**5. Lenguaje natural (Javier).** Se recomienda usar el modelo de lenguaje solo como borrador, con las transacciones de la alerta en el contexto y temperatura 0.2. Esta configuración reduce la alucinación de 15% a 5% y ahorra 4.5 minutos por narrativa (75%). Como el 5% de las narrativas aún contiene hechos inventados, cada cifra citada se verifica de forma automática contra la secuencia del remitente. El oficial de cumplimiento revisa y firma cada narrativa y responde por su contenido.

**6. Explicabilidad (Chet).** Se recomienda entregar explicaciones SHAP, cuyo error de eficiencia máximo es de 3e-07 log-odds, al oficial de cumplimiento y al regulador. No se deben entregar LIME con 500 muestras, porque solo coincide en 0.45 entre corridas, ni los pesos de atención, que se correlacionan 0.35 con SHAP. Al cliente señalado no se le entrega ninguna explicación del modelo, para no enseñarle a evadir el sistema. Antes del lanzamiento se debe verificar la estabilidad de SHAP con distintos conjuntos de fondo.

**7. Operaciones (Mejía).** Con el umbral U2, el piloto generó 250 alertas diarias frente a una capacidad de 40, es decir, 6.25 veces lo que el equipo puede revisar. Revisarlas todas cuesta USD 3,000 diarios y detecta 10 de los 12 casos esperados (83.3%), pero revisar 40 al azar detecta solo 1.6. Se recomienda no operar con U2 y el equipo actual, ordenar la cola por puntaje y ajustar umbral y capacidad para que las alertas quepan. Un caso no detectado (USD 50,000) cuesta lo mismo que 4,167 revisiones, lo que justifica ampliar la capacidad.

**Cifras que no cuadraron.**
- Precisión de U2 de 0.95 frente a 4% en el piloto. La primera se midió con 50% de positivos. Proyectada a 0.2%, U2 da 3.85% y 249 alertas diarias, igual que el piloto. Se usa la tabla proyectada.
- AUC de 0.87 frente a 0.81. La primera usa partición aleatoria con fuga entre ventanas. Se usa 0.81 y se considera optimista el recall de la tabla de umbrales.

**Crítica de la IA.** La extracción del PDF con IA devolvió ilegible el valor de SHAP, que se recuperó decodificando el documento (3 × 10⁻⁷). La IA estimó unas 10 alertas diarias con la precisión balanceada, cuando la proyección real es de 249. También recomendó U1 por costo total sin considerar que requiere 122 analistas.

**Defensa ante el regulador.** Ante la autoridad reguladora se presentaría la validación con partición temporal (AUC de 0.81 en abril y mayo), la tabla de umbrales proyectada a la prevalencia real y la explicación SHAP de cada alerta, con un error de eficiencia de 3e-07. Se reconoce como riesgo residual que con U3 se esperan 4.2 casos no detectados por día, cifra que puede ser mayor porque el recall proviene de una partición aleatoria. La empresa se compromete a medir cada mes el recall, la precisión y las alertas frente a la capacidad, y a recalibrar el umbral si se incumple alguna condición. Responde por la decisión el oficial de cumplimiento, con la aprobación de la gerencia general.

---

### c. Aporte de cada integrante

> **Borrador.** Cada integrante debe ajustar sus tres líneas con su experiencia real.

| Integrante | Aporte | Corrección a la IA y verificación | Dato de otro rol |
|---|---|---|---|
| Majo | Sección 1. Diagnóstico de sobreajuste y BCE ponderada. | La IA tomó 0.95 como precisión real. Se recalculó con prevalencia de 0.2% y dio 3.85%. | Partición temporal (Rol 3). |
| Tunchez | Sección 2. Elección del ajuste fino de dos bloques. | La IA prefirió el ajuste completo por su menor pérdida de entrenamiento. Se verificó con la tabla que su AUC es menor (0.84 frente a 0.87). | AUC temporal de 0.81 (Rol 3). |
| Dulce | Sección 3. Fuga por partición aleatoria y ventana de 30 días. | La IA presentó la exactitud de 99.8% como buen resultado. Se verificó que un sistema que nunca alerta obtiene esa misma cifra. | Tabla de umbrales (Rol 1). |
| Nadissa | Sección 4. Uso condicionado de difusión. | La IA propuso medir el clasificador con datos sintéticos. Se corrigió para probar solo con datos reales de abril y mayo. | AUC temporal base de 0.81 (Rol 3). |
| Javier | Sección 5. Condiciones para el modelo de lenguaje. | La IA reportó 6% de alucinación al leer "6 de 40". Se recalculó 6/40 y dio 15%. | Volumen de alertas diarias (Rol 7). |
| Chet | Sección 6. Explicación por audiencia. | La IA propuso los pesos de atención como explicación. Se verificó que su correlación con SHAP es solo 0.35. | Tipologías de lavado (Rol 4). |
| Mejía | Sección 7 y Task 2.2. Capacidad y elección de U3. | La IA eligió U1 por menor costo total. Se verificó que requiere 122 analistas. | Recall y tasa de falsos positivos (Rol 1). |

### d. Entrega

El memo de la Task 2.4b se exporta a PDF y se sube a Canvas cuando el docente lo indique.

## Task 3. Presentación y defensa

**3.1a.** El esquema de una página está en `presentacion/esquema.excalidraw` y se abre en [excalidraw.com](https://excalidraw.com). Contiene la decisión, el umbral y la capacidad, la cifra que más pesó y el principal riesgo.

**3.1b y 3.2.** Son actividades presenciales. Para el ensayo cronometrado de 2 minutos se sugiere este orden.

1. Decisión, umbral y capacidad (30 segundos).
2. Cifras reconciliadas, 0.95 frente a 3.85% y 0.87 frente a 0.81 (45 segundos).
3. Principal riesgo y condiciones de lanzamiento (30 segundos).
4. Dónde falló la IA (15 segundos).

Como el docente sortea a quién pregunta, cada integrante debe poder explicar la cifra clave y la recomendación de las siete secciones con la tabla de la Task 2.1a.